# Smart Vehicle Security Monitoring System Using Computer Vision

Team Members:
- Shog Aloufi: car and person detection using YOLO and OpenCV
- Rahaf Altir: distance estimation and suspicious behavior detection
- Shahad Nasser: accident detection and alert generation

This notebook contains the full final project code in one place. All videos and `yolov8s.pt` are in the same folder as this notebook, so file names are used directly.

## Part 1: Car and Person Detection

This part uses YOLO to detect cars and people. OpenCV reads the video frame by frame, draws bounding boxes, displays labels and confidence scores, counts people and cars, and saves the output video.

In [ ]:
# Import libraries
import cv2
from ultralytics import YOLO

In [ ]:
# Load YOLO model
model = YOLO("yolov8s.pt")

In [ ]:
# Part 1 - Test video 1
input_video = "normal_case1.mp4"
output_video = "normal_case1_detection.mp4"

cap = cv2.VideoCapture(input_video)

if not cap.isOpened():
    print("Error: Could not open video.")
else:
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)

    if fps == 0:
        fps = 25

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    out = cv2.VideoWriter(output_video, fourcc, fps, (width, height))

    while True:
        ret, frame = cap.read()

        if not ret:
            break

        results = model(frame, verbose=False)

        person_count = 0
        car_count = 0

        for result in results:
            boxes = result.boxes

            for box in boxes:
                class_id = int(box.cls[0])
                confidence = float(box.conf[0])

                # YOLO class 0 = person, class 2 = car
                if (class_id == 0 and confidence >= 0.30) or (class_id == 2 and confidence >= 0.45):
                    x1, y1, x2, y2 = box.xyxy[0]
                    x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)

                    if class_id == 0:
                        label = "Person"
                        color = (0, 255, 0)
                        person_count = person_count + 1
                    else:
                        label = "Car"
                        color = (255, 0, 0)
                        car_count = car_count + 1

                    cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                    cv2.putText(frame, label + " " + str(round(confidence, 2)),
                                (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX,
                                0.6, color, 2)

        # Display counters and close time
        cv2.putText(frame, "Persons: " + str(person_count),
                    (25, height - 75), cv2.FONT_HERSHEY_SIMPLEX,
                    0.7, (0, 255, 0), 2)
        cv2.putText(frame, "Cars: " + str(car_count),
                    (25, height - 45), cv2.FONT_HERSHEY_SIMPLEX,
                    0.7, (255, 0, 0), 2)

        out.write(frame)
        cv2.imshow("Part 1 - normal_case1", frame)

        if cv2.waitKey(30) & 0xFF == ord('q'):
            break

    cap.release()
    out.release()
    cv2.destroyAllWindows()

    print("Detection completed successfully.")
    print("Output saved as", output_video)

## Part 2: Distance and Suspicious Behavior Detection

This part uses the detected car and person boxes. The largest car is treated as the parked vehicle. The left side of the car box is used as a simple driver-side area. If a person stays close for 12 seconds, the alert appears.

In [ ]:
# Part 2 - Suspicious behavior detection
input_video = "suspicious_case7.mp4"
output_video = "suspicious_case7_output.mp4"

suspicious_time_threshold = 12

cap = cv2.VideoCapture(input_video)

if not cap.isOpened():
    print("Error: Could not open video.")
else:
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)

    if fps == 0:
        fps = 25

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    out = cv2.VideoWriter(output_video, fourcc, fps, (width, height))

    close_frame_count = 0

    while True:
        ret, frame = cap.read()

        if not ret:
            break

        results = model(frame, verbose=False)

        car_boxes = []
        person_boxes = []

        for result in results:
            boxes = result.boxes

            for box in boxes:
                class_id = int(box.cls[0])
                confidence = float(box.conf[0])

                if class_id == 0 and confidence >= 0.30:
                    x1, y1, x2, y2 = box.xyxy[0]
                    person_boxes.append((int(x1), int(y1), int(x2), int(y2), confidence))

                if class_id == 2 and confidence >= 0.45:
                    x1, y1, x2, y2 = box.xyxy[0]
                    car_boxes.append((int(x1), int(y1), int(x2), int(y2), confidence))

        for car in car_boxes:
            x1, y1, x2, y2, confidence = car
            cv2.rectangle(frame, (x1, y1), (x2, y2), (255, 0, 0), 2)
            cv2.putText(frame, "Car " + str(round(confidence, 2)),
                        (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX,
                        0.6, (255, 0, 0), 2)

        for person in person_boxes:
            x1, y1, x2, y2, confidence = person
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(frame, "Person " + str(round(confidence, 2)),
                        (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX,
                        0.6, (0, 255, 0), 2)

        person_is_close = False

        if len(car_boxes) > 0 and len(person_boxes) > 0:
            # Choose the largest car as the main vehicle
            largest_area = 0
            main_car = car_boxes[0]

            for car in car_boxes:
                car_area = (car[2] - car[0]) * (car[3] - car[1])
                if car_area > largest_area:
                    largest_area = car_area
                    main_car = car

            car_x1, car_y1, car_x2, car_y2, car_conf = main_car
            car_width = car_x2 - car_x1
            car_height = car_y2 - car_y1

            driver_x1 = max(0, car_x1 - car_width // 2)
            driver_y1 = max(0, car_y1 - car_height // 3)
            driver_x2 = min(width, car_x1 + car_width // 2)
            driver_y2 = min(height, car_y2 + car_height // 3)

            cv2.rectangle(frame, (driver_x1, driver_y1), (driver_x2, driver_y2), (0, 255, 255), 2)
            cv2.putText(frame, "Driver-side area", (driver_x1, driver_y1 - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)

            car_center_x = (car_x1 + car_x2) // 2
            car_center_y = (car_y1 + car_y2) // 2

            for person in person_boxes:
                px1, py1, px2, py2, p_conf = person
                person_center_x = (px1 + px2) // 2
                person_center_y = (py1 + py2) // 2

                distance = ((person_center_x - car_center_x) ** 2 + (person_center_y - car_center_y) ** 2) ** 0.5
                close_distance = car_width * 0.9

                overlap_driver_side = px1 < driver_x2 and px2 > driver_x1 and py1 < driver_y2 and py2 > driver_y1

                if distance < close_distance or overlap_driver_side:
                    person_is_close = True
                    cv2.line(frame, (person_center_x, person_center_y),
                             (car_center_x, car_center_y), (0, 255, 255), 2)
                    cv2.putText(frame, "Close to vehicle", (px1, py2 + 25),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)

        if person_is_close:
            close_frame_count = close_frame_count + 1
        else:
            close_frame_count = 0

        close_seconds = close_frame_count / fps

        cv2.putText(frame, "Close time: " + str(round(close_seconds, 1)) + " sec",
                    (25, height - 25), cv2.FONT_HERSHEY_SIMPLEX,
                    0.7, (0, 255, 255), 2)

        if close_seconds >= suspicious_time_threshold:
            cv2.putText(frame, "Suspicious Behavior Detected",
                        (30, 60), cv2.FONT_HERSHEY_SIMPLEX,
                        1.0, (0, 0, 255), 3)

        out.write(frame)
        cv2.imshow("Part 2 - Suspicious Behavior", frame)

        if cv2.waitKey(30) & 0xFF == ord('q'):
            break

    cap.release()
    out.release()
    cv2.destroyAllWindows()

    print("Suspicious behavior detection completed successfully.")
    print("Output saved as", output_video)

## Part 3: Accident Detection

This part uses frame difference. If there is large abnormal movement near a detected vehicle for several frames, the alert appears.

In [ ]:
# Part 3 - Accident detection
input_video = "accident_case3.mp4"
output_video = "accident_case3_output.mp4"

cap = cv2.VideoCapture(input_video)

if not cap.isOpened():
    print("Error: Could not open video.")
else:
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)

    if fps == 0:
        fps = 25

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    out = cv2.VideoWriter(output_video, fourcc, fps, (width, height))

    previous_gray = None
    accident_counter = 0

    while True:
        ret, frame = cap.read()

        if not ret:
            break

        results = model(frame, verbose=False)
        car_boxes = []

        for result in results:
            boxes = result.boxes

            for box in boxes:
                class_id = int(box.cls[0])
                confidence = float(box.conf[0])

                if class_id == 2 and confidence >= 0.40:
                    x1, y1, x2, y2 = box.xyxy[0]
                    car_boxes.append((int(x1), int(y1), int(x2), int(y2), confidence))

        for car in car_boxes:
            x1, y1, x2, y2, confidence = car
            cv2.rectangle(frame, (x1, y1), (x2, y2), (255, 0, 0), 2)
            cv2.putText(frame, "Car " + str(round(confidence, 2)),
                        (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX,
                        0.6, (255, 0, 0), 2)

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY) # gray is faster
        gray = cv2.GaussianBlur(gray, (21, 21), 0)     # reduce the noise to clac the stranger motion

        abnormal_motion = False

        if previous_gray is not None:
            frame_difference = cv2.absdiff(previous_gray, gray)
            threshold = cv2.threshold(frame_difference, 25, 255, cv2.THRESH_BINARY)[1]
            threshold = cv2.dilate(threshold, None, iterations=2)

            contours, hierarchy = cv2.findContours(threshold.copy(),
                                                   cv2.RETR_EXTERNAL,
                                                   cv2.CHAIN_APPROX_SIMPLE)

            for contour in contours:
                motion_area = cv2.contourArea(contour)

                if motion_area > 2500:
                    x, y, w, h = cv2.boundingRect(contour)
                    motion_center_x = x + w // 2
                    motion_center_y = y + h // 2

                    for car in car_boxes:
                        car_x1, car_y1, car_x2, car_y2, car_conf = car
                        margin_x = (car_x2 - car_x1) // 2
                        margin_y = (car_y2 - car_y1) // 2

                        danger_x1 = max(0, car_x1 - margin_x)
                        danger_y1 = max(0, car_y1 - margin_y)
                        danger_x2 = min(width, car_x2 + margin_x)
                        danger_y2 = min(height, car_y2 + margin_y)

                        inside_danger_area = motion_center_x >= danger_x1 and motion_center_x <= danger_x2 and motion_center_y >= danger_y1 and motion_center_y <= danger_y2

                        if inside_danger_area:
                            abnormal_motion = True
                            cv2.rectangle(frame, (x, y), (x + w, y + h),
                                          (0, 255, 255), 2)
                            cv2.putText(frame, "Abnormal motion", (x, y - 10),
                                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)
                            break

        previous_gray = gray

        if abnormal_motion:
            accident_counter = accident_counter + 1
        else:
            if accident_counter > 0:
                accident_counter = accident_counter - 1

        if accident_counter >= 3:
            cv2.putText(frame, "Possible Accident Detected",
                        (30, 60), cv2.FONT_HERSHEY_SIMPLEX,
                        1.0, (0, 0, 255), 3)

        out.write(frame)
        cv2.imshow("Part 3 - Accident Detection", frame)

        if cv2.waitKey(30) & 0xFF == ord('q'):
            break

    cap.release()
    out.release()
    cv2.destroyAllWindows()

    print("Accident detection completed successfully.")
    print("Output saved as", output_video)

## Final Demo

lab-style demo. Change `input_video` to a video name, or set `input_video = 0` to open the camera.


In [ ]:
# Final Demo - Simple Lab Style
# Put a video name, or put 0 to open the camera.

input_video = "accident_case3.mp4"   # example: "normal_case1.mp4", "suspicious_case7.mp4", or 0
cap = cv2.VideoCapture(input_video)

if input_video == 0:
    output_video = "camera_final_demo_output.mp4"
else:
    output_video = input_video.replace(".mp4", "_final_demo_output.mp4")

if not cap.isOpened():
    print("Error: Could not open video or camera.")
else:
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)

    if fps == 0:
        fps = 25

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    out = cv2.VideoWriter(output_video, fourcc, fps, (width, height))

    suspicious_time_threshold = 12
    close_counter = 0
    accident_counter = 0
    previous_gray = None

    while True:
        ret, frame = cap.read()

        if not ret:
            break

        results = model(frame, verbose=False)

        person_count = 0
        car_count = 0
        person_boxes = []
        car_boxes = []

        for result in results:
            boxes = result.boxes

            for box in boxes:
                class_id = int(box.cls[0])
                confidence = float(box.conf[0])

                if class_id == 0 and confidence >= 0.30:
                    x1, y1, x2, y2 = box.xyxy[0]
                    x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)

                    person_count = person_count + 1
                    person_boxes.append((x1, y1, x2, y2))

                    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                    cv2.putText(frame, "Person " + str(round(confidence, 2)),
                                (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX,
                                0.6, (0, 255, 0), 2)

                if class_id == 2 and confidence >= 0.45:
                    x1, y1, x2, y2 = box.xyxy[0]
                    x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)

                    car_count = car_count + 1
                    car_boxes.append((x1, y1, x2, y2))

                    cv2.rectangle(frame, (x1, y1), (x2, y2), (255, 0, 0), 2)
                    cv2.putText(frame, "Car " + str(round(confidence, 2)),
                                (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX,
                                0.6, (255, 0, 0), 2)

        # Count the final boxes detected in this frame
        person_count = len(person_boxes)
        car_count = len(car_boxes)

        # Display counters and close time
        cv2.putText(frame, "Persons: " + str(person_count),
                    (25, height - 75), cv2.FONT_HERSHEY_SIMPLEX,
                    0.7, (0, 255, 0), 2)
        cv2.putText(frame, "Cars: " + str(car_count),
                    (25, height - 45), cv2.FONT_HERSHEY_SIMPLEX,
                    0.7, (255, 0, 0), 2)

        # Suspicious behavior: person close to car for 12 seconds
        person_close = False

        for person in person_boxes:
            px1, py1, px2, py2 = person
            person_center_x = (px1 + px2) // 2
            person_center_y = (py1 + py2) // 2

            for car in car_boxes:
                cx1, cy1, cx2, cy2 = car
                car_center_x = (cx1 + cx2) // 2
                car_center_y = (cy1 + cy2) // 2
                car_width = cx2 - cx1

                distance = ((person_center_x - car_center_x) ** 2 +
                            (person_center_y - car_center_y) ** 2) ** 0.5

                if distance < car_width:
                    person_close = True
                    cv2.line(frame, (person_center_x, person_center_y),
                             (car_center_x, car_center_y), (0, 255, 255), 2)
                    cv2.putText(frame, "Close to vehicle", (px1, py2 + 25),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)

        if person_close:
            close_counter = close_counter + 1
        else:
            close_counter = 0

        close_time = close_counter / fps

        cv2.putText(frame, "Close time: " + str(round(close_time, 1)) + " sec",
                    (25, height - 15), cv2.FONT_HERSHEY_SIMPLEX,
                    0.7, (0, 255, 255), 2)

        if close_time >= suspicious_time_threshold:
            cv2.putText(frame, "Suspicious Behavior Detected",
                        (30, 70), cv2.FONT_HERSHEY_SIMPLEX,
                        1.0, (0, 0, 255), 3)

        # Accident detection: simple frame difference
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        gray = cv2.GaussianBlur(gray, (21, 21), 0)

        accident_detected = False

        if previous_gray is not None:
            frame_diff = cv2.absdiff(previous_gray, gray)
            thresh = cv2.threshold(frame_diff, 25, 255, cv2.THRESH_BINARY)[1]
            thresh = cv2.dilate(thresh, None, iterations=2)

            contours, hierarchy = cv2.findContours(thresh.copy(),
                                                   cv2.RETR_EXTERNAL,
                                                   cv2.CHAIN_APPROX_SIMPLE)

            for contour in contours:
                area = cv2.contourArea(contour)

                if area > 2500:
                    x, y, w, h = cv2.boundingRect(contour)
                    motion_center_x = x + w // 2
                    motion_center_y = y + h // 2

                    for car in car_boxes:
                        cx1, cy1, cx2, cy2 = car

                        if motion_center_x >= cx1 and motion_center_x <= cx2 and motion_center_y >= cy1 and motion_center_y <= cy2:
                            accident_detected = True
                            cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 255), 2)
                            cv2.putText(frame, "Abnormal motion", (x, y - 10),
                                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)

        previous_gray = gray

        if accident_detected:
            accident_counter = accident_counter + 1
        else:
            accident_counter = 0

        if accident_counter >= 3:
            cv2.putText(frame, "Possible Accident Detected",
                        (30, 115), cv2.FONT_HERSHEY_SIMPLEX,
                        1.0, (0, 0, 255), 3)

        out.write(frame)
        cv2.imshow("Final Demo", frame)

        if cv2.waitKey(30) & 0xFF == ord('q'):
            break

    cap.release()
    out.release()
    cv2.destroyAllWindows()

    print("Final demo completed successfully.")
    print("Output saved as", output_video)


## Testing Results

Tested videos:
- `normal_case1.mp4`
- `suspicious_case7.mp4`
- `accident_case3.mp4`

Saved outputs are created in the same folder as this notebook.